In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model


def build_control_state_model(
    frame_size=(224, 224, 3),
    feat_hw=(7, 7),
    feat_c=256,
    timesteps=128,
    num_classes=3,
):
    """
    Two-input, two-output model:
      Input1: RGB frame (H,W,3) -> Conv2D -> (7,7,256)
      Input2: Internal state (T,7,7,256) for Conv3D (depth=T)
      Concat: (T+1,7,7,256) -> Conv3D -> heads:
        - classification logits (num_classes)
        - next internal state (T,7,7,256)

    Notes:
      - Conv3D in TF expects (batch, depth, height, width, channels)
      - timesteps corresponds to depth axis
    """

    # -------------------------
    # Input 1: RGB frame
    # -------------------------
    frame_in = layers.Input(shape=frame_size, name="frame_rgb")  # (224,224,3)

    # A simple Conv2D encoder that ends at (7,7,256)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(frame_in)
    x = layers.MaxPool2D()(x)  # 112
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D()(x)  # 56
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D()(x)  # 28
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D()(x)  # 14
    x = layers.Conv2D(feat_c, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D()(x)  # 7

    # Ensure exact shape (7,7,256) even if you adjust earlier layers
    frame_feat = layers.Resizing(feat_hw[0], feat_hw[1], name="frame_feat_resize")(x)
    frame_feat = layers.Conv2D(feat_c, 1, padding="same", activation=None, name="frame_feat_proj")(frame_feat)
    # frame_feat: (7,7,256)

    # Expand to depth=1 so it can be concatenated with the internal state depth axis
    frame_feat_3d = layers.Lambda(lambda t: tf.expand_dims(t, axis=1), name="frame_feat_expand_depth")(frame_feat)
    # frame_feat_3d: (batch, 1, 7, 7, 256)

    # -------------------------
    # Input 2: internal state
    # -------------------------
    state_in = layers.Input(shape=(timesteps, feat_hw[0], feat_hw[1], feat_c), name="state_in")
    # state_in: (batch, T, 7, 7, 256)

    # -------------------------
    # Concat along depth axis: (T + 1, 7, 7, 256)
    # -------------------------
    volume = layers.Concatenate(axis=1, name="concat_state_and_frame")([state_in, frame_feat_3d])
    # volume: (batch, T+1, 7, 7, 256)

    # -------------------------
    # Further 3D CNN processing
    # -------------------------
    v = layers.Conv3D(256, kernel_size=(3, 3, 3), padding="same", activation="relu")(volume)
    v = layers.BatchNormalization()(v)
    v = layers.Conv3D(256, kernel_size=(3, 3, 3), padding="same", activation="relu")(v)
    v = layers.BatchNormalization()(v)

    # Optional: slightly mix across time more explicitly
    v = layers.Conv3D(256, kernel_size=(3, 1, 1), padding="same", activation="relu")(v)

    # -------------------------
    # Output 1: classification head
    # -------------------------
    cls = layers.GlobalAveragePooling3D()(v)         # (batch, 256)
    cls = layers.Dense(256, activation="relu")(cls)
    cls_logits = layers.Dense(num_classes, name="action_logits")(cls)  # logits for Left/None/Right

    # -------------------------
    # Output 2: updated internal state (T,7,7,256)
    # -------------------------
    # We want to output a tensor with depth=T (not T+1).
    # A simple design: drop the oldest slice and keep the newest T slices after processing.
    # Here we keep the last T slices: v[:, 1:, ...] gives shape (batch, T, 7,7,256)
    next_state = layers.Lambda(lambda t: t[:, 1:, :, :, :], name="next_state_slice")(v)

    # Ensure channel count is exactly feat_c for the feedback state
    next_state = layers.Conv3D(feat_c, kernel_size=(1, 1, 1), padding="same", activation=None, name="next_state_proj")(next_state)
    # next_state: (batch, T, 7, 7, 256)

    model = Model(
        inputs=[frame_in, state_in],
        outputs=[cls_logits, next_state],
        name="ControlStateNet",
    )
    return model


In [2]:
model = build_control_state_model(timesteps=128, feat_hw=(7, 7), feat_c=256, num_classes=3)
model.summary()

Model: "ControlStateNet"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 frame_rgb (InputLayer)         [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 224, 224, 32  896         ['frame_rgb[0][0]']              
                                )                                                                 
                                                                                                  
 max_pooling2d (MaxPooling2D)   (None, 112, 112, 32  0           ['conv2d[0][0]']                 
                                )                                                   

In [ ]:
state = tf.zeros((1, 128, 7, 7, 256), dtype=tf.float32)

# frame should be float32 in [0,1] or whatever you train with
frame = tf.zeros((1, 224, 224, 3), dtype=tf.float32)

logits, state = model([frame, state], training=False)
print(logits.shape, state.shape)  # (1,3) and (1,128,7,7,256)

(1, 3) (1, 128, 7, 7, 256)


In [4]:
state

<tf.Tensor: shape=(1, 128, 7, 7, 256), dtype=float32, numpy=
array([[[[[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         ...,

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ..

In [14]:
import csv
from pathlib import Path
import tensorflow as tf


def _make_target_onehot(left: int, right: int):
    # y = [Left, None, Right]
    none = 1 if (left == right) else 0  # (0,0)->1 and (1,1)->1
    return tf.constant([left, none, right], dtype=tf.float32)


def train_supervised_with_context(
    model,
    frames_dir: str,
    csv_path: str,
    epochs: int = 1,
    lr: float = 1e-4,
    timesteps: int = 128,
    feat_hw=(7, 7),
    feat_c: int = 256,
    image_size=(224, 224),
    print_every: int = 50,
    clipnorm: float = 1.0,
):
    """
    Train the model as supervised learning WITH context.
    - We feed the current (frame, state) and train on the action label.
    - We treat the state as external: gradients DO NOT flow through time.
    - After each step, we update state = stop_gradient(next_state).

    This is exactly: supervised learning with a context input.
    """

    frames_dir = Path(frames_dir)
    csv_path = Path(csv_path)

    # Read and sort rows so frames are in order
    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows = sorted(reader, key=lambda r: r["filename"])

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=clipnorm)
    loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        # Start each epoch with zero state (traditional and predictable)
        state = tf.zeros((1, timesteps, feat_hw[0], feat_hw[1], feat_c), dtype=tf.float32)

        running_loss = 0.0
        running_correct = 0
        seen = 0

        for i, r in enumerate(rows):
            fname = r["filename"]
            left = int(r["left"])
            right = int(r["right"])

            # ---- Load frame and normalize /255 ----
            img_path = frames_dir / fname
            img_bytes = tf.io.read_file(str(img_path))
            img = tf.image.decode_jpeg(img_bytes, channels=3)       # uint8
            img = tf.image.convert_image_dtype(img, tf.float32)     # float32 [0,1] (div by 255)
            img = tf.image.resize(img, image_size)
            frame = tf.expand_dims(img, axis=0)                     # (1,H,W,3)

            # ---- Target one-hot ----
            y = _make_target_onehot(left, right)                    # (3,)
            y = tf.expand_dims(y, axis=0)                           # (1,3)
            y_idx = int(tf.argmax(y, axis=-1)[0].numpy())           # 0,1,2

            # Treat state as external input (no time gradients)
            state = tf.stop_gradient(state)

            # ---- One supervised training step ----
            with tf.GradientTape() as tape:
                logits, next_state = model([frame, state], training=True)
                loss = loss_fn(y, logits)

            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

            # ---- Update state (detached) ----
            state = tf.stop_gradient(next_state)

            # ---- Metrics ----
            pred_idx = int(tf.argmax(logits, axis=-1)[0].numpy())
            running_correct += int(pred_idx == y_idx)
            running_loss += float(loss.numpy())
            seen += 1

            if i == 0 or (i + 1) % print_every == 0:
                avg_loss = running_loss / seen
                acc = running_correct / seen
                none_val = int(left == right)
                print(
                    f"[{i+1}/{len(rows)}] {fname} | "
                    f"target(L,N,R)=({left},{none_val},{right}) -> y={y_idx} | "
                    f"pred={pred_idx} | "
                    f"loss={float(loss.numpy()):.4f} avg_loss={avg_loss:.4f} acc={acc:.4f}"
                )

        print(f"Epoch {epoch} done | avg_loss={running_loss/seen:.4f} acc={running_correct/seen:.4f}")

    return model


In [15]:
model = train_supervised_with_context(
    model=model,
    frames_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101",
    csv_path=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101_controls_extracted.csv",
    epochs=1,
    lr=1e-4,
    timesteps=128,
    feat_hw=(7, 7),
    feat_c=256,
    image_size=(224, 224),
    print_every=2,
)


Epoch 1/1
[1/151485] frame_000000_t_000000000000us.jpg | target(L,N,R)=(0,1,0) -> y=1 | pred=1 | loss=0.4077 avg_loss=0.4077 acc=1.0000
[2/151485] frame_000001_t_000000017041us.jpg | target(L,N,R)=(0,1,0) -> y=1 | pred=1 | loss=0.1206 avg_loss=0.2641 acc=1.0000
[4/151485] frame_000003_t_000000051124us.jpg | target(L,N,R)=(0,1,0) -> y=1 | pred=1 | loss=0.0825 avg_loss=0.1755 acc=1.0000
[6/151485] frame_000005_t_000000085208us.jpg | target(L,N,R)=(0,1,0) -> y=1 | pred=1 | loss=0.0752 avg_loss=0.1410 acc=1.0000
[8/151485] frame_000007_t_000000119291us.jpg | target(L,N,R)=(0,1,0) -> y=1 | pred=1 | loss=0.0401 avg_loss=0.1162 acc=1.0000
[10/151485] frame_000009_t_000000153374us.jpg | target(L,N,R)=(0,1,0) -> y=1 | pred=1 | loss=0.0241 avg_loss=0.0986 acc=1.0000
[12/151485] frame_000011_t_000000187458us.jpg | target(L,N,R)=(0,1,0) -> y=1 | pred=1 | loss=0.0155 avg_loss=0.0852 acc=1.0000
[14/151485] frame_000013_t_000000221541us.jpg | target(L,N,R)=(0,1,0) -> y=1 | pred=1 | loss=0.0109 avg_l

KeyboardInterrupt: 

In [16]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_2layer_3dcnn_t10_to_t2(
    height=64,
    width=64,
    channels=3,
    base_filters=32,
    out_filters=16,
):
    """
    2-layer 3D CNN:
    Input:  (B, 10, H, W, C)
    Output: (B, 2,  H/4, W/4, out_filters)  (because we stride in space too)
    """

    x_in = layers.Input(shape=(10, height, width, channels), name="video_in")

    # Layer 1: downsample time 10 -> 5, and space H/W -> H/2, W/2
    x = layers.Conv3D(
        filters=base_filters,
        kernel_size=(3, 3, 3),
        strides=(2, 2, 2),      # (time, height, width)
        padding="same",
        activation="relu",
        name="conv3d_1",
    )(x_in)

    # Layer 2: downsample time 5 -> 2 (with stride 2 and SAME padding gives ceil(5/2)=3, so we crop to 2)
    x = layers.Conv3D(
        filters=out_filters,
        kernel_size=(3, 3, 3),
        strides=(2, 2, 2),
        padding="same",
        activation="relu",
        name="conv3d_2",
    )(x)

    # After conv3d_2, time is typically ceil(5/2)=3, so force it to exactly 2 timesteps
    # Crop 1 timestep from the end: T=3 -> T=2
    x = layers.Cropping3D(cropping=((0, 1), (0, 0), (0, 0)), name="crop_to_2t")(x)

    model = Model(inputs=x_in, outputs=x, name="two_layer_3dcnn_t10_to_t2")
    return model


model = build_2layer_3dcnn_t10_to_t2(height=64, width=64, channels=3)
model.summary()


Model: "two_layer_3dcnn_t10_to_t2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 video_in (InputLayer)       [(None, 10, 64, 64, 3)]   0         
                                                                 
 conv3d_1 (Conv3D)           (None, 5, 32, 32, 32)     2624      
                                                                 
 conv3d_2 (Conv3D)           (None, 3, 16, 16, 16)     13840     
                                                                 
 crop_to_2t (Cropping3D)     (None, 2, 16, 16, 16)     0         
                                                                 
Total params: 16,464
Trainable params: 16,464
Non-trainable params: 0
_________________________________________________________________


In [12]:
trained_model = train_stateful_from_csv(
    model=model,
    frames_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101",
    csv_path=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101_controls_extracted.csv",
    epochs=1,
    lr=1e-4,
    timesteps=128,
    feat_hw=(7, 7),
    feat_c=256,
    image_size=(224, 224),
    unroll_steps=16,
    print_every_updates=50,
    clipnorm=1.0,
)



Epoch 1/1
Update 000001 | seen_frames=16 | loss=0.8104 avg_loss=0.8104 | acc=1.0000 avg_acc=1.0000
Update 000050 | seen_frames=800 | loss=0.0088 avg_loss=0.1485 | acc=1.0000 avg_acc=1.0000
Update 000100 | seen_frames=1600 | loss=0.8236 avg_loss=0.4780 | acc=0.7500 avg_acc=0.8831
Update 000150 | seen_frames=2400 | loss=0.0431 avg_loss=0.4885 | acc=1.0000 avg_acc=0.8733
Update 000200 | seen_frames=3200 | loss=0.6922 avg_loss=0.6315 | acc=0.7500 avg_acc=0.8153
Update 000250 | seen_frames=4000 | loss=0.1649 avg_loss=0.6940 | acc=1.0000 avg_acc=0.7927
Update 000300 | seen_frames=4800 | loss=0.7972 avg_loss=0.7316 | acc=0.7500 avg_acc=0.7719
Update 000350 | seen_frames=5600 | loss=1.0224 avg_loss=0.7354 | acc=0.7500 avg_acc=0.7718
Update 000400 | seen_frames=6400 | loss=5.3955 avg_loss=0.7289 | acc=0.0000 avg_acc=0.7786


KeyboardInterrupt: 

In [9]:
import csv
from pathlib import Path
import tensorflow as tf


def _make_target(left: int, right: int):
    """
    Convert (left,right) -> one-hot [Left, None, Right]
    Rules you gave:
      - if left==0 and right==0 => none=1
      - if left==1 and right==1 => none=1 (ambiguous, treat as none)
      - otherwise none=0
    """
    none = 1 if (left == right) else 0  # covers (0,0) and (1,1)
    # one-hot [Left, None, Right]
    return tf.constant([left, none, right], dtype=tf.float32)


def run_stateful_sequence_from_csv(
    model,
    frames_dir: str,
    csv_path: str,
    timesteps: int = 128,
    feat_hw=(7, 7),
    feat_c: int = 256,
):
    """
    Reads CSV rows (filename,left,right) in order, loads frames, normalizes /255,
    runs model(frame, state) sequentially, and updates state from model output.

    Returns:
      results: list of dicts with filename, target, pred_class, probs (optional)
    """

    frames_dir = Path(frames_dir)
    csv_path = Path(csv_path)

    # Initial internal state: zeros for the first frame
    state = tf.zeros((1, timesteps, feat_hw[0], feat_hw[1], feat_c), dtype=tf.float32)

    results = []

    # Read and sort rows by filename to ensure sequential order
    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows = sorted(reader, key=lambda r: r["filename"])

    for i, r in enumerate(rows):
        fname = r["filename"]
        left = int(r["left"])
        right = int(r["right"])

        # ---- load image and normalize /255 -> [0,1] ----
        img_path = frames_dir / fname
        img_bytes = tf.io.read_file(str(img_path))
        img = tf.image.decode_jpeg(img_bytes, channels=3)  # uint8
        img = tf.image.convert_image_dtype(img, tf.float32)  # float32 in [0,1] (div by 255)
        img = tf.image.resize(img, (224, 224))  # match your model's expected input
        frame = tf.expand_dims(img, axis=0)  # (1,224,224,3)

        # ---- target: [Left, None, Right] ----
        y = _make_target(left, right)          # (3,)
        y = tf.expand_dims(y, axis=0)          # (1,3)

        # ---- forward pass with state feedback ----
        logits, next_state = model([frame, state], training=False)

        probs = tf.nn.softmax(logits, axis=-1)
        pred_idx = int(tf.argmax(probs, axis=-1)[0].numpy())  # 0=Left,1=None,2=Right

        results.append({
            "filename": fname,
            "target_left": left,
            "target_none": int((left == right)),
            "target_right": right,
            "pred_idx": pred_idx,
            "pred_probs": probs[0].numpy(),
        })

        # ---- update state for next frame ----
        state = next_state
        if i == 0 or (i + 1) % 50 == 0:
            print(
                f"[{i+1}/{len(rows)}] "
                f"{fname} | "
                f"target(L,N,R)=({left},{int(left==right)},{right}) | "
                f"pred={pred_idx}"
            )

    return results


In [10]:
results = run_stateful_sequence_from_csv(
    model=model,
    frames_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101",
    csv_path=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_224x101_controls_extracted.csv",
    timesteps=128,
    feat_hw=(7, 7),
    feat_c=256,
)
results[0]


[1/151485] frame_000000_t_000000000000us.jpg | target(L,N,R)=(0,1,0) | pred=0
[50/151485] frame_000049_t_000000835040us.jpg | target(L,N,R)=(0,1,0) | pred=1
[100/151485] frame_000099_t_000001687122us.jpg | target(L,N,R)=(0,1,0) | pred=0
[150/151485] frame_000149_t_000002539204us.jpg | target(L,N,R)=(0,1,0) | pred=1
[200/151485] frame_000199_t_000003391287us.jpg | target(L,N,R)=(0,1,0) | pred=0
[250/151485] frame_000249_t_000004243369us.jpg | target(L,N,R)=(0,1,0) | pred=0
[300/151485] frame_000299_t_000005095451us.jpg | target(L,N,R)=(0,1,0) | pred=1
[350/151485] frame_000349_t_000005947533us.jpg | target(L,N,R)=(0,1,0) | pred=0
[400/151485] frame_000399_t_000006799616us.jpg | target(L,N,R)=(0,1,0) | pred=0
[450/151485] frame_000449_t_000007651698us.jpg | target(L,N,R)=(0,1,0) | pred=0
[500/151485] frame_000499_t_000008503780us.jpg | target(L,N,R)=(0,1,0) | pred=0
[550/151485] frame_000549_t_000009355862us.jpg | target(L,N,R)=(0,1,0) | pred=0
[600/151485] frame_000599_t_000010207944us.

KeyboardInterrupt: 